# Includes

In [16]:
import pandas as pd
import numpy as np
import mne
import mne_nirs
import h5py
import shutil
import plotly.express as px

from pathlib import Path
from mne.preprocessing.nirs import source_detector_distances, short_channels, optical_density, beer_lambert_law

root = Path.home() / "fnirs-representation-learning"
rs_data_dir = root / "snirf_dataset_2"

In [17]:
clean_file_rows = []

for subject_dir in sorted(rs_data_dir.glob("Subj*")):
    if not subject_dir.is_dir():
        continue

    resting_file = subject_dir / "resting.snirf"
    clean_file = subject_dir / "resting_clean.snirf"

    if not clean_file.exists():
        shutil.copy2(resting_file, clean_file)

        with h5py.File(clean_file, "r+") as f:
            if "stim1" in f["nirs"]:
                del f["nirs"]["stim1"]

        print("created:", clean_file)
        
    else:
        print("already exists:", clean_file)

    clean_file_rows.append({
        "subject": subject_dir.name,
        "clean_file": clean_file,
    })

clean_files_df = pd.DataFrame(clean_file_rows)
clean_files_df

already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj92/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learnin

,subject,clean_file
0,Subj100,/home/asunkari/fnirs-representation-learning/s...
1,Subj101,/home/asunkari/fnirs-representation-learning/s...
2,Subj102,/home/asunkari/fnirs-representation-learning/s...
3,Subj103,/home/asunkari/fnirs-representation-learning/s...
4,Subj104,/home/asunkari/fnirs-representation-learning/s...
5,Subj86,/home/asunkari/fnirs-representation-learning/s...
6,Subj91,/home/asunkari/fnirs-representation-learning/s...
7,Subj92,/home/asunkari/fnirs-representation-learning/s...
8,Subj94,/home/asunkari/fnirs-representation-learning/s...
9,Subj95,/home/asunkari/fnirs-representation-learning/s...


In [18]:
subject_summary_rows = []
pair_rows = []

for row in clean_files_df.itertuples(index=False):
    subject = row.subject
    clean_file = row.clean_file

    raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)

    # pick all fNIRS channels, then keep only CW amplitude channels
    picks_fnirs = mne.pick_types(raw_rest.info, fnirs=True)
    channel_types = np.array(raw_rest.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]

    # distances and SS/LS masks
    dists = source_detector_distances(raw_rest.info, picks=picks_cw)

    ss_mask_all = short_channels(raw_rest.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_cw]

    ls_mask = dists >= 0.025

    # channel-level names
    cw_names = np.array(raw_rest.ch_names)[picks_cw]
    pair_names = np.array([name.split(" ")[0] for name in cw_names])

    # pair-level table for this subject
    pair_table = pd.DataFrame({
        "subject": subject,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": dists,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    pair_summary = (
        pair_table.groupby(["subject", "pair_name"], as_index=False)
        .agg(
            distance_m=("distance_m", "first"),
            is_ss=("is_ss", "first"),
            is_ls=("is_ls", "first"),
        )
    )

    pair_summary["group"] = np.select(
        [pair_summary["is_ss"], pair_summary["is_ls"]],
        ["SS", "LS"],
        default="MID"
    )

    pair_rows.append(pair_summary)

    group_counts = pair_summary["group"].value_counts()

    subject_summary_rows.append({
        "subject": subject,
        "file": str(clean_file),
        "sfreq": raw_rest.info["sfreq"],
        "duration_s": raw_rest.times[-1],
        "n_fnirs_channels": len(picks_fnirs),
        "n_cw_channels": len(picks_cw),
        "distance_min_m": float(dists.min()),
        "distance_max_m": float(dists.max()),
        "n_ss_channels": int(ss_mask.sum()),
        "n_ls_channels": int(ls_mask.sum()),
        "n_pairs_total": len(pair_summary),
        "n_pairs_ss": int(group_counts.get("SS", 0)),
        "n_pairs_ls": int(group_counts.get("LS", 0)),
        "n_pairs_mid": int(group_counts.get("MID", 0)),
    })

subject_summary_df = pd.DataFrame(subject_summary_rows)
pair_summary_df = pd.concat(pair_rows, ignore_index=True)

subject_summary_df

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf


...

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj96/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj97/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj98/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj99/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



,subject,file,sfreq,duration_s,n_fnirs_channels,n_cw_channels,distance_min_m,distance_max_m,n_ss_channels,n_ls_channels,n_pairs_total,n_pairs_ss,n_pairs_ls,n_pairs_mid
0,Subj100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,0.008,0.030463,16,96,56,8,48,0
1,Subj101,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
2,Subj102,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
3,Subj103,/home/asunkari/fnirs-representation-learning/s...,50.0,716.98,112,112,0.008,0.030463,16,96,56,8,48,0
4,Subj104,/home/asunkari/fnirs-representation-learning/s...,50.0,669.98,112,112,0.008,0.030463,16,96,56,8,48,0
5,Subj86,/home/asunkari/fnirs-representation-learning/s...,50.0,590.98,112,112,0.008,0.030463,16,96,56,8,48,0
6,Subj91,/home/asunkari/fnirs-representation-learning/s...,50.0,641.98,112,112,0.008,0.030463,16,96,56,8,48,0
7,Subj92,/home/asunkari/fnirs-representation-learning/s...,50.0,588.98,112,112,0.008,0.030463,16,96,56,8,48,0
8,Subj94,/home/asunkari/fnirs-representation-learning/s...,50.0,692.98,112,112,0.008,0.030463,16,96,56,8,48,0
9,Subj95,/home/asunkari/fnirs-representation-learning/s...,50.0,765.98,112,112,0.008,0.030463,16,96,56,8,48,0


# Inspect semisynthetic HRF files and event annotations

In [19]:
output_dir = root / "outputs"
output_tables_dir = output_dir / "tables"
output_figures_dir = output_dir / "figures"

output_tables_dir.mkdir(parents=True, exist_ok=True)
output_figures_dir.mkdir(parents=True, exist_ok=True)

file_labels = [
    ("clean", "resting_clean.snirf"),
    ("hrf_20", "resting_hrf_20.snirf"),
    ("hrf_50", "resting_hrf_50.snirf"),
    ("hrf_100", "resting_hrf_100.snirf"),
]

In [20]:
def get_cw_channel_indices(raw_snirf):
    picks_fnirs = mne.pick_types(raw_snirf.info, fnirs=True)
    channel_types = np.array(raw_snirf.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]
    return picks_cw


def build_channel_table(raw_snirf, subject_name, file_label):
    picks_cw = get_cw_channel_indices(raw_snirf)
    cw_names = np.array(raw_snirf.ch_names)[picks_cw]

    dists = source_detector_distances(raw_snirf.info, picks=picks_cw)

    ss_mask_all = short_channels(raw_snirf.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_cw]

    ls_mask = dists >= 0.025
    pair_names = np.array([name.split(" ")[0] for name in cw_names])

    channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": dists,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    channel_table["group"] = np.select(
        [channel_table["is_ss"], channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return channel_table


def build_event_table(raw_snirf, subject_name, file_label):
    annotations = raw_snirf.annotations

    event_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "onset_s": annotations.onset,
        "duration_s": annotations.duration,
        "description": annotations.description,
    })

    return event_table

In [ ]:
subject_name = "Subj100"
subject_dir = rs_data_dir / subject_name

subject_file_rows = []
subject_event_tables = []
subject_channel_tables = []

for file_label, file_name in file_labels:
    file_path = subject_dir / file_name

    if not file_path.exists():
        print("missing:", file_path)
        continue

    raw_snirf = mne.io.read_raw_snirf(file_path, preload=False, verbose=False)
    channel_table = build_channel_table(raw_snirf, subject_name, file_label)
    event_table = build_event_table(raw_snirf, subject_name, file_label)

    subject_file_rows.append({
        "subject": subject_name,
        "file_label": file_label,
        "file_path": str(file_path),
        "sfreq": raw_snirf.info["sfreq"],
        "duration_s": raw_snirf.times[-1],
        "n_channels_total": len(raw_snirf.ch_names),
        "n_cw_channels": len(channel_table),
        "n_pairs": channel_table["pair_name"].nunique(),
        "n_ss_pairs": int((channel_table.groupby("pair_name")["group"].first() == "SS").sum()),
        "n_ls_pairs": int((channel_table.groupby("pair_name")["group"].first() == "LS").sum()),
        "n_mid_pairs": int((channel_table.groupby("pair_name")["group"].first() == "MID").sum()),
        "n_annotations": len(raw_snirf.annotations),
        "annotation_descriptions": "|".join(sorted(set(raw_snirf.annotations.description))),
    })

    subject_channel_tables.append(channel_table)
    subject_event_tables.append(event_table)

subject_file_summary_df = pd.DataFrame(subject_file_rows)
subject_channel_summary_df = pd.concat(subject_channel_tables, ignore_index=True)
subject_event_summary_df = pd.concat(subject_event_tables, ignore_index=True)

subject_file_summary_df

/tmp/ipykernel_384858/64686942.py:16: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/64686942.py:16: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/64686942.py:16: RuntimeWarning:

The data only

,subject,file_label,file_path,sfreq,duration_s,n_channels_total,n_cw_channels,n_pairs,n_ss_pairs,n_ls_pairs,n_mid_pairs,n_annotations,annotation_descriptions
0,Subj100,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,0,
1,Subj100,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
2,Subj100,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
3,Subj100,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1


In [ ]:
subject_event_summary_df

,subject,file_label,onset_s,duration_s,description
0,Subj100,hrf_20,1.22,1.0,1
1,Subj100,hrf_20,21.06,1.0,1
2,Subj100,hrf_20,41.42,1.0,1
3,Subj100,hrf_20,61.06,1.0,1
4,Subj100,hrf_20,82.66,1.0,1
...,...,...,...,...,...
103,Subj100,hrf_100,621.06,1.0,1
104,Subj100,hrf_100,641.42,1.0,1
105,Subj100,hrf_100,661.82,1.0,1
106,Subj100,hrf_100,680.22,1.0,1


In [ ]:
annotation_figure = px.scatter(
    subject_event_summary_df,
    x="onset_s",
    y="description",
    color="file_label",
    hover_data=["duration_s"],
    title=f"{subject_name}: annotation overview",
)

annotation_figure.show()
annotation_figure.write_html(output_figures_dir / f"{subject_name.lower()}_annotation_overview.html")

In [ ]:
overlay_rows = []

for file_label, file_name in file_labels:
    file_path = subject_dir / file_name

    if not file_path.exists():
        continue

    raw_snirf = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)
    channel_table = build_channel_table(raw_snirf, subject_name, file_label)

    long_channel_names = channel_table.loc[channel_table["group"] == "LS", "channel_name"].tolist()
    selected_channel_names = long_channel_names[:8]

    if len(selected_channel_names) == 0:
        continue

    selected_channel_indices = mne.pick_channels(raw_snirf.ch_names, selected_channel_names)
    selected_channel_data = raw_snirf.get_data(picks=selected_channel_indices)

    mean_signal = selected_channel_data.mean(axis=0)

    overlay_rows.append(pd.DataFrame({
        "time_s": raw_snirf.times,
        "signal": mean_signal,
        "file_label": file_label,
    }))

overlay_df = pd.concat(overlay_rows, ignore_index=True)
overlay_df.head()

/tmp/ipykernel_384858/1585496126.py:10: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/1585496126.py:10: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/1585496126.py:10: RuntimeWarning:

The dat

,time_s,signal,file_label
0,0.00,333787.709133,clean
1,0.02,332026.431202,clean
2,0.04,332342.541776,clean
3,0.06,332298.859065,clean
4,0.08,330981.611509,clean


In [ ]:
overlay_figure = px.line(
    overlay_df,
    x="time_s",
    y="signal",
    color="file_label",
    title=f"{subject_name}: mean LS CW amplitude overlay",
)

overlay_figure.show()
overlay_figure.write_html(output_figures_dir / f"{subject_name.lower()}_ls_overlay.html")

In [ ]:
all_file_rows = []
all_event_tables = []
all_channel_tables = []

for subject_dir in sorted(rs_data_dir.glob("Subj*")):
    if not subject_dir.is_dir():
        continue

    subject_name = subject_dir.name

    for file_label, file_name in file_labels:
        file_path = subject_dir / file_name

        if not file_path.exists():
            print("missing:", file_path)
            continue

        raw_snirf = mne.io.read_raw_snirf(file_path, preload=False, verbose=False)
        channel_table = build_channel_table(raw_snirf, subject_name, file_label)
        event_table = build_event_table(raw_snirf, subject_name, file_label)

        pair_groups = channel_table.groupby("pair_name")["group"].first()

        all_file_rows.append({
            "subject": subject_name,
            "file_label": file_label,
            "file_path": str(file_path),
            "sfreq": raw_snirf.info["sfreq"],
            "duration_s": raw_snirf.times[-1],
            "n_channels_total": len(raw_snirf.ch_names),
            "n_cw_channels": len(channel_table),
            "n_pairs": channel_table["pair_name"].nunique(),
            "n_ss_pairs": int((pair_groups == "SS").sum()),
            "n_ls_pairs": int((pair_groups == "LS").sum()),
            "n_mid_pairs": int((pair_groups == "MID").sum()),
            "n_annotations": len(raw_snirf.annotations),
            "annotation_descriptions": "|".join(sorted(set(raw_snirf.annotations.description))),
        })

        all_channel_tables.append(channel_table)
        all_event_tables.append(event_table)

all_file_summary_df = pd.DataFrame(all_file_rows)
all_channel_summary_df = pd.concat(all_channel_tables, ignore_index=True)
all_event_summary_df = pd.concat(all_event_tables, ignore_index=True)

/tmp/ipykernel_384858/2952689692.py:19: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2952689692.py:19: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2952689692.py:19: RuntimeWarning:

The dat

In [ ]:
all_file_summary_df

,subject,file_label,file_path,sfreq,duration_s,n_channels_total,n_cw_channels,n_pairs,n_ss_pairs,n_ls_pairs,n_mid_pairs,n_annotations,annotation_descriptions
0,Subj100,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,0,
1,Subj100,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
2,Subj100,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
3,Subj100,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
4,Subj101,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,0,
5,Subj101,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
6,Subj101,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
7,Subj101,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
8,Subj102,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,0,
9,Subj102,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1


In [ ]:
all_file_summary_df.groupby("file_label")[["n_annotations", "n_ss_pairs", "n_ls_pairs", "n_mid_pairs"]].mean(numeric_only=True)

,n_annotations,n_ss_pairs,n_ls_pairs,n_mid_pairs
file_label,,,,
clean,0.0,8.0,48.0,0.0
hrf_100,34.5,8.0,48.0,0.0
hrf_20,34.5,8.0,48.0,0.0
hrf_50,34.5,8.0,48.0,0.0


In [ ]:
all_event_summary_df.groupby(["file_label", "description"]).size().reset_index(name="count")

,file_label,description,count
0,hrf_100,1,483
1,hrf_20,1,483
2,hrf_50,1,483


In [ ]:
subject_name = "Subj100"
subject_dir = rs_data_dir / subject_name

file_label = "hrf_50"
file_name = "resting_hrf_50.snirf"
file_path = subject_dir / file_name

raw_cw = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)
print(raw_cw)
print(raw_cw.annotations)

<RawSNIRF | resting_hrf_50.snirf, 112 x 36800 (736.0 s), ~31.6 MB, data loaded>
<Annotations | 36 segments: 1 (36)>


/tmp/ipykernel_384858/1670690465.py:9: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



In [ ]:
raw_od = optical_density(raw_cw.copy())
raw_hb = beer_lambert_law(raw_od, ppf=0.1)

print(raw_hb)
print(raw_hb.get_channel_types()[:10])
print(raw_hb.ch_names[:10])

<RawSNIRF | resting_hrf_50.snirf, 112 x 36800 (736.0 s), ~31.6 MB, data loaded>
['hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo', 'hbo']
['S1_D1 hbo', 'S1_D7 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D5 hbo', 'S2_D7 hbo', 'S2_D8 hbo', 'S3_D2 hbo', 'S3_D3 hbo', 'S3_D8 hbo']


In [ ]:
pd.Series(raw_hb.get_channel_types()).value_counts()

hbo    56
hbr    56
Name: count, dtype: int64

In [ ]:
def build_hb_channel_table(raw_hb, subject_name, file_label):
    picks_hbo = mne.pick_types(raw_hb.info, fnirs="hbo")
    picks_hbr = mne.pick_types(raw_hb.info, fnirs="hbr")
    picks_hb = np.sort(np.concatenate([picks_hbo, picks_hbr]))

    hb_names = np.array(raw_hb.ch_names)[picks_hb]
    hb_types = np.array(raw_hb.get_channel_types())[picks_hb]
    pair_names = np.array([channel_name.split(" ")[0] for channel_name in hb_names])

    distances_all = source_detector_distances(raw_hb.info)
    distances_hb = distances_all[picks_hb]

    ss_mask_all = short_channels(raw_hb.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_hb]
    ls_mask = distances_hb >= 0.025

    hb_channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": hb_names,
        "pair_name": pair_names,
        "chromophore": hb_types,
        "distance_m": distances_hb,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    hb_channel_table["group"] = np.select(
        [hb_channel_table["is_ss"], hb_channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return hb_channel_table

In [ ]:
hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)
hb_channel_table.head()

,subject,file_label,channel_name,pair_name,chromophore,distance_m,is_ss,is_ls,group
0,Subj100,hrf_50,S1_D1 hbo,S1_D1,hbo,0.030017,False,True,LS
1,Subj100,hrf_50,S1_D7 hbo,S1_D7,hbo,0.030017,False,True,LS
2,Subj100,hrf_50,S2_D1 hbo,S2_D1,hbo,0.030017,False,True,LS
3,Subj100,hrf_50,S2_D2 hbo,S2_D2,hbo,0.030017,False,True,LS
4,Subj100,hrf_50,S2_D5 hbo,S2_D5,hbo,0.008000,True,False,SS


In [ ]:
ls_hbo_channel_names = hb_channel_table.loc[
    (hb_channel_table["group"] == "LS") & (hb_channel_table["chromophore"] == "hbo"),
    "channel_name",
].tolist()

ls_hbr_channel_names = hb_channel_table.loc[
    (hb_channel_table["group"] == "LS") & (hb_channel_table["chromophore"] == "hbr"),
    "channel_name",
].tolist()

selected_hbo_channel_names = ls_hbo_channel_names[:8]
selected_hbr_channel_names = ls_hbr_channel_names[:8]

print("selected HbO channels:", selected_hbo_channel_names)
print("selected HbR channels:", selected_hbr_channel_names)

selected HbO channels: ['S1_D1 hbo', 'S1_D7 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D7 hbo', 'S2_D8 hbo', 'S3_D2 hbo', 'S3_D3 hbo']
selected HbR channels: ['S1_D1 hbr', 'S1_D7 hbr', 'S2_D1 hbr', 'S2_D2 hbr', 'S2_D7 hbr', 'S2_D8 hbr', 'S3_D2 hbr', 'S3_D3 hbr']


In [ ]:
events, event_id = mne.events_from_annotations(raw_hb, verbose=False)

print("event_id:", event_id)
print("events shape:", events.shape)
events[:10]

event_id: {'1': 1}
events shape: (36, 3)


array([[ 157,    0,    1],
       [1016,    0,    1],
       [2095,    0,    1],
       [3075,    0,    1],
       [4109,    0,    1],
       [5098,    0,    1],
       [6040,    0,    1],
       [7019,    0,    1],
       [8002,    0,    1],
       [9011,    0,    1]])

In [ ]:
epochs_hb = mne.Epochs(
    raw_hb,
    events=events,
    event_id=event_id,
    tmin=-5.0,
    tmax=20.0,
    baseline=(-2.0, 0.0),
    preload=True,
    detrend=None,
    verbose=False,
)

print(epochs_hb)

<Epochs |  35 events (all good), -5 - 20 sec, baseline -2 – 0 sec, ~37.5 MB, data loaded,
 '1': 35>


In [ ]:
epochs_hbo = epochs_hb.copy().pick(selected_hbo_channel_names)
epochs_hbr = epochs_hb.copy().pick(selected_hbr_channel_names)

hbo_epoch_data = epochs_hbo.get_data()
hbr_epoch_data = epochs_hbr.get_data()

mean_hbo_time_course = hbo_epoch_data.mean(axis=(0, 1))
mean_hbr_time_course = hbr_epoch_data.mean(axis=(0, 1))

epoch_time = epochs_hbo.times

mean_epoch_df = pd.DataFrame({
    "time_s": np.concatenate([epoch_time, epoch_time]),
    "signal": np.concatenate([mean_hbo_time_course, mean_hbr_time_course]),
    "chromophore": ["HbO"] * len(epoch_time) + ["HbR"] * len(epoch_time),
})

mean_epoch_df.head()

,time_s,signal,chromophore
0,-5.00,-0.000003,HbO
1,-4.98,-0.000003,HbO
2,-4.96,-0.000002,HbO
3,-4.94,-0.000002,HbO
4,-4.92,-0.000002,HbO


In [ ]:
mean_epoch_figure = px.line(
    mean_epoch_df,
    x="time_s",
    y="signal",
    color="chromophore",
    title=f"{subject_name} {file_label}: mean event-locked Hb response",
)

mean_epoch_figure.add_vline(x=0.0)
mean_epoch_figure.show()

In [ ]:
amplitude_epoch_rows = []

for file_label, file_name in [("hrf_20", "resting_hrf_20.snirf"),
                              ("hrf_50", "resting_hrf_50.snirf"),
                              ("hrf_100", "resting_hrf_100.snirf")]:

    file_path = subject_dir / file_name
    raw_cw = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)

    raw_od = optical_density(raw_cw.copy())
    raw_hb = beer_lambert_law(raw_od, ppf=0.1)

    hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)

    ls_hbo_channel_names = hb_channel_table.loc[
        (hb_channel_table["group"] == "LS") & (hb_channel_table["chromophore"] == "hbo"),
        "channel_name",
    ].tolist()

    ls_hbr_channel_names = hb_channel_table.loc[
        (hb_channel_table["group"] == "LS") & (hb_channel_table["chromophore"] == "hbr"),
        "channel_name",
    ].tolist()

    selected_hbo_channel_names = ls_hbo_channel_names[:8]
    selected_hbr_channel_names = ls_hbr_channel_names[:8]

    events, event_id = mne.events_from_annotations(raw_hb, verbose=False)

    epochs_hb = mne.Epochs(
        raw_hb,
        events=events,
        event_id=event_id,
        tmin=-5.0,
        tmax=20.0,
        baseline=(-2.0, 0.0),
        preload=True,
        detrend=None,
        verbose=False,
    )

    hbo_epoch_data = epochs_hb.copy().pick(selected_hbo_channel_names).get_data()
    hbr_epoch_data = epochs_hb.copy().pick(selected_hbr_channel_names).get_data()

    mean_hbo_time_course = hbo_epoch_data.mean(axis=(0, 1))
    mean_hbr_time_course = hbr_epoch_data.mean(axis=(0, 1))

    amplitude_epoch_rows.append(pd.DataFrame({
        "time_s": epochs_hb.times,
        "signal": mean_hbo_time_course,
        "chromophore": "HbO",
        "file_label": file_label,
    }))

    amplitude_epoch_rows.append(pd.DataFrame({
        "time_s": epochs_hb.times,
        "signal": mean_hbr_time_course,
        "chromophore": "HbR",
        "file_label": file_label,
    }))

amplitude_epoch_df = pd.concat(amplitude_epoch_rows, ignore_index=True)
amplitude_epoch_df.head()

/tmp/ipykernel_384858/2956190080.py:9: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2956190080.py:9: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2956190080.py:9: RuntimeWarning:

The data o

,time_s,signal,chromophore,file_label
0,-5.00,-0.000006,HbO,hrf_20
1,-4.98,-0.000007,HbO,hrf_20
2,-4.96,-0.000007,HbO,hrf_20
3,-4.94,-0.000007,HbO,hrf_20
4,-4.92,-0.000006,HbO,hrf_20


In [ ]:
amplitude_epoch_figure = px.line(
    amplitude_epoch_df,
    x="time_s",
    y="signal",
    color="file_label",
    facet_row="chromophore",
    title=f"{subject_name}: mean event-locked Hb responses across amplitudes",
)

amplitude_epoch_figure.add_vline(x=0.0)
amplitude_epoch_figure.show()
amplitude_epoch_figure.write_html(output_figures_dir / f"{subject_name.lower()}_hb_epoch_amplitude_comparison.html")